# Protocol B: fixed-total sweep: imports and config hash

In [1]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("KAGGLE_KEY")
secret_value_1 = user_secrets.get_secret("KAGGLE_USERNAME")

In [2]:
import sys
import os
import glob
import shutil
import warnings
import numpy as np
import pandas as pd
import torch

warnings.filterwarnings(
    "ignore",
    message="The behavior of DataFrame concatenation with empty or all-NA entries is deprecated",
    category=FutureWarning,
)

try:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _sec.get_secret("KAGGLE_KEY")
    print("kaggle credentials loaded from secrets")
except Exception as exc:
    print("kaggle secrets not loaded:", exc)

try:
    import core
except ModuleNotFoundError:
    hits = glob.glob("/kaggle/input/**/core.py", recursive=True)
    if not hits:
        raise ModuleNotFoundError("core.py not found.")
    sys.path.insert(0, os.path.dirname(hits[0]))
    import core

print("import-time CONFIG_HASH:", core.CONFIG_HASH)

kaggle credentials loaded from secrets
import-time CONFIG_HASH: 03fa342b535795c1


# GPU guard

In [3]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "No CUDA device. Protocol B trains ResNet-18 across 54 new runs; "
        "a CPU kernel is infeasible. Switch the Kaggle accelerator to GPU (P100) and rerun. "
        "Notebook 00 can run on CPU, notebook 02 cannot."
    )
print("CUDA available:", torch.cuda.get_device_name(0))

CUDA available: Tesla T4


# Stage the frozen config, refresh, then load data

In [4]:
if core.DATAWISE_RESULTS_SLUG is not None:
    try:
        core.pull_results_store()
        print("pulled results store from", core.DATAWISE_RESULTS_SLUG)
    except Exception as exc:
        print("pull skipped (first session or empty store):", exc)

frozen_target = core.PATHS["frozen_config"]
if not os.path.exists(frozen_target):
    hits = glob.glob("/kaggle/input/**/frozen_config.md", recursive=True)
    if not hits:
        raise FileNotFoundError(
            "frozen_config.md not found in the working dir or any mounted dataset. "
            "Run 00_setup.ipynb and mount the results store (or the dataset holding "
            "frozen_config.md) before running this notebook."
        )
    core._ensure(os.path.dirname(frozen_target))
    shutil.copy(hits[0], frozen_target)
    print("staged frozen_config.md from", hits[0])

h = core.refresh_config()
print("active CONFIG_HASH after refresh:", h)
print("core.NORM_MEAN now:", core.NORM_MEAN)
print("core.NORM_STD  now:", core.NORM_STD)

data = core.load_pcam()

Dataset URL: https://www.kaggle.com/datasets/hrushisanap/datawise-results
pulled results store from hrushisanap/datawise-results
active CONFIG_HASH after refresh: 03fa342b535795c1
core.NORM_MEAN now: [0.700756, 0.538358, 0.691621]
core.NORM_STD  now: [0.234978, 0.277409, 0.212895]
[03:23:42] train pos/neg 131072/131072
[03:23:42] valid pos/neg (raw) 16369/16399
[03:23:42] test pos/neg (raw) 16377/16391
[03:23:43] valid balanced 32738 (16369/16369)
[03:23:43] test balanced 32754 (16377/16377)


# Build the Protocol B condition list

In [5]:
n_pos = len(data.train_pos_idx)
n_neg = len(data.train_neg_idx)

conditions = []
print(f"{'idx':>3}  {'target r':>8}  {'n_min':>7}  {'n_maj':>7}  {'total N':>8}  {'actual r':>8}")
for i, r in enumerate(core.RATIOS):
    n_min = int(round(core.N_F / (1 + r)))
    n_maj = core.N_F - n_min
    actual_ratio = n_maj / n_min
    assert n_min + n_maj == core.N_F, f"total {n_min + n_maj} != N_F {core.N_F} at r={r}"
    assert n_min <= n_pos, f"n_min {n_min} exceeds positive pool {n_pos} at r={r}"
    assert n_maj <= n_neg, f"n_maj {n_maj} exceeds negative pool {n_neg} at r={r}"
    conditions.append((r, n_min, n_maj))
    print(f"{i:>3}  {r:>8}  {n_min:>7}  {n_maj:>7}  {n_min + n_maj:>8}  {actual_ratio:>8.2f}")

print(f"\n{len(conditions)} conditions, {len(core.MODEL_NAMES)} models, "
      f"{len(core.SEEDS)} seeds -> "
      f"{len(conditions) * len(core.MODEL_NAMES) * len(core.SEEDS)} run_training calls")

idx  target r    n_min    n_maj   total N  actual r
  0         1    32000    32000     64000      1.00
  1         2    21333    42667     64000      2.00
  2         5    10667    53333     64000      5.00
  3        10     5818    58182     64000     10.00
  4        25     2462    61538     64000     25.00
  5        50     1255    62745     64000     50.00
  6       100      634    63366     64000     99.95

7 conditions, 3 models, 3 seeds -> 63 run_training calls


# Run the sweep (models outermost, then conditions, then seeds)

In [6]:
done = 0
for model in core.MODEL_NAMES:
    for (r, n_min, n_maj) in conditions:
        for s in core.SEEDS:
            run_id = core.run_training(
                "protocolB", model, n_min, n_maj, s, s, s, data,
                target_ratio=r, resume=True)
            done += 1
            print(f"[{done:>2}/63] {model} r={r} m={n_min} M={n_maj} seed={s} -> {run_id}")

print("sweep calls completed:", done)

[03:23:43] [protocolB][logistic][m=32000 M=32000 r=1.00][seed=21] reuse of logistic__m32000_M32000__d21_i21_l21, no retrain
[ 1/63] logistic r=1 m=32000 M=32000 seed=21 -> logistic__m32000_M32000__d21_i21_l21
[03:23:43] [protocolB][logistic][m=32000 M=32000 r=1.00][seed=22] reuse of logistic__m32000_M32000__d22_i22_l22, no retrain
[ 2/63] logistic r=1 m=32000 M=32000 seed=22 -> logistic__m32000_M32000__d22_i22_l22
[03:23:43] [protocolB][logistic][m=32000 M=32000 r=1.00][seed=23] reuse of logistic__m32000_M32000__d23_i23_l23, no retrain
[ 3/63] logistic r=1 m=32000 M=32000 seed=23 -> logistic__m32000_M32000__d23_i23_l23
[03:23:43] logistic parameter count: 55298
[03:23:53] [protocolB][logistic][m=21333 M=42667 r=2.00][seed=21] step 250/5000 loss=0.5426
[03:24:02] [protocolB][logistic][m=21333 M=42667 r=2.00][seed=21] step 500/5000 loss=0.5661
[03:24:11] [protocolB][logistic][m=21333 M=42667 r=2.00][seed=21] step 750/5000 loss=0.4381
[03:24:20] [protocolB][logistic][m=21333 M=42667 r=2.0

# Session summary from the ledger (no test metrics)

In [7]:
if not os.path.exists(core.PATHS["ledger"]):
    raise FileNotFoundError("no ledger yet; run the sweep cell first")

ledger = pd.read_csv(core.PATHS["ledger"])
mask = ledger["protocol"].astype(str).str.contains("protocolB")
rows = ledger[mask]

order = {"failed": 0, "success": 1}
rows = rows.assign(_rank=rows["status"].map(order).fillna(-1))
dedup = rows.sort_values("_rank").drop_duplicates("run_id", keep="last").drop(columns="_rank")
n_success = int((dedup["status"] == "success").sum())
n_failed = int((dedup["status"] == "failed").sum())
expected = len(conditions) * len(core.MODEL_NAMES) * len(core.SEEDS)

print(f"Protocol B: {n_success} success, {n_failed} failed, "
      f"{expected - n_success} pending of {expected}")

failed_rows = dedup[dedup["status"] == "failed"]
if len(failed_rows):
    print("\nfailed runs:")
    for _, r in failed_rows.iterrows():
        print(f"  {r['run_id']}  (model={r['model']} "
              f"n_min={r['n_min']} n_maj={r['n_maj']} seed={r['data_seed']})")

core._ensure(core.PATHS["summaries"])
summary_path = os.path.join(core.PATHS["summaries"], "02_protocolB.md")
lines = [
    "# Notebook 02: Protocol B (fixed total) session summary",
    "",
    f"Active config hash: {core.CONFIG_HASH}",
    "",
    f"- expected unique runs: {expected} (7 conditions x 3 models x 3 seeds)",
    f"- success: {n_success}",
    f"- failed: {n_failed}",
    f"- pending: {expected - n_success}",
    "- the r=1 point (32000, 32000) is shared with Protocols A and D, so of the 63 "
    "expected unique runs 9 are reuses (3 models x 3 seeds) and this protocol adds 54 new runs.",
    "",
    "## Failed runs",
]
if len(failed_rows):
    for _, r in failed_rows.iterrows():
        lines.append(f"- {r['run_id']} (model={r['model']} "
                     f"n_min={r['n_min']} n_maj={r['n_maj']} seed={r['data_seed']})")
else:
    lines.append("- none")
lines.append("")
with open(summary_path, "w") as f:
    f.write("\n".join(lines))
print("\nwrote", summary_path)

Protocol B: 63 success, 0 failed, 0 pending of 63

wrote /kaggle/working/summaries/02_protocolB.md


# Persistence

In [8]:
if core.DATAWISE_RESULTS_SLUG is not None:
    core.push_results_store()
    print("pushed results store to", core.DATAWISE_RESULTS_SLUG)
else:
    print("DATAWISE_RESULTS_SLUG is unset; persistence skipped this session.")
    print("WARNING: without it, the ledger and logits will not survive session end.")

Starting upload for file frozen_config.md


100%|██████████| 431/431 [00:00<00:00, 1.48kB/s]


Upload successful: frozen_config.md (431B)
Starting upload for file condition_registry.csv


100%|██████████| 495/495 [00:00<00:00, 2.10kB/s]


Upload successful: condition_registry.csv (495B)
Starting upload for file random_permutations.zip


...

Upload successful: logits.zip (36MB)
Starting upload for file results.zip


100%|██████████| 2.01k/2.01k [00:00<00:00, 8.78kB/s]


Upload successful: results.zip (2KB)
Starting upload for file training_curves.zip


100%|██████████| 47.0k/47.0k [00:00<00:00, 220kB/s]


Upload successful: training_curves.zip (47KB)
pushed results store to hrushisanap/datawise-results
